In [13]:
import os
import ast
import json
from typing import List

import pandas as pd
from tqdm import tqdm


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "data_dir": "../mall",
            "process_label_file": "../mall/label_process.csv",
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
            "step_span": 10,
            "logs_padding": ["No related logs"],
        }
    elif dataset == "train-ticket":
        return {
            "data_dir": "../train-ticket",
            "process_label_file": "../train-ticket/label_process.csv",
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
            "step_span": 10,
            "logs_padding": ["No related logs"],
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class DataAligner:
    """
    作用：
    1. 读取 label_process.csv 和 ip_metrics.csv
    2. 对每条 fault 记录关联 trace / logs / metrics
    3. 生成 data_alignment_idx_{idx}.parquet
    4. 拼接成 data_alignment_1.parquet
    """

    def __init__(self, config: dict):
        self.config = config
        self.data_dir = config["data_dir"]
        self.label_file_url = config["process_label_file"]
        self.ip_metrics_url = os.path.join(self.data_dir, "ip_metrics.csv")
        self.step_span = config["step_span"]
        self.outer_name = config["outer_name"]
        self.logs_padding = config["logs_padding"]

        self.label_df: pd.DataFrame | None = None
        self.ip_metrics_df: pd.DataFrame | None = None

    def load_inputs(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        self.label_df = pd.read_csv(self.label_file_url)
        '''
            fault            start_time    end_time     file_name \                                  
            CPU overload     (s)             (s)      "['./trace/file1.parquet', './trace/file2.parquet']"
            Service error    (s)             (s)

            fault_id    trace_id_list                       count            span_len
            0           "['traceA', 'traceB', 'traceC']"    trace_id 的数量   每个 trace 的长度（span数量）
        '''
        self.ip_metrics_df = pd.read_csv(self.ip_metrics_url)
        '''
            timestamp	IpAddress	host_cpu_usage	pod_memory_usage
            1000	10.0.0.1	0.5	0.7
            1001	10.0.0.1	0.6	0.8
            1000	10.0.0.2	0.4	0.6
        '''
        return self.label_df, self.ip_metrics_df

    @staticmethod
    def parse_list_string(value):
        """
        把 CSV 里像 "['a', 'b']" 这种字符串转回 list。
        """
        if isinstance(value, str):
            return ast.literal_eval(value)
        return value

    def build_trace_and_log_tables(self, row: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        读取当前 fault 对应的 trace parquet 和 log parquet，并拼接。
        """
        df_merge_list = []
        log_df_merge_list = []

        file_name_list = self.parse_list_string(row["file_name"])

        for filename in file_name_list:
            # 日志名和trace名的后缀一样
            log_file_name = filename.replace("trace/trace_info", "logs/logs")
            '''
                 Duration  HaveStack LogEventList                      OperationName  \
                0        29      False           []  /components/api/v1/mall/user_info   
                1        28      False           []            MallController.userInfo   
                
                       ParentSpanId ResultCode RpcId  RpcType   ServiceIp   ServiceName  \
                0                 0         -1              0  10.0.0.176  mall-gateway   
                1  1aaf1265717fa2db         -1              0  10.0.0.176  mall-gateway   
                
                             SpanId                                       TagEntryList  \
                0  1aaf1265717fa2db  [{"Key":"app","Value":"mall-gateway"},{"Key":"...   
                1  5ec8742aec5925dd  [{"Key":"app","Value":"mall-gateway"},{"Key":"...   
                
                       Timestamp                           TraceID  
                0  1711595187228  4d1f3abd3e3683e34d6acd44c1ca144f  
                1  1711595187229  4d1f3abd3e3683e34d6acd44c1ca144f
            '''
            df_merge_list.append(pd.read_parquet(filename))
            '''
                                                             content  \
                0  2024-03-28 11:40:32.319 [org.springframework.k...   
                1  2024-03-28 11:40:32.319 [org.springframework.k...   
                
                                                _time_ _source_  \
                0  2024-03-28T11:40:32.320038888+08:00   stdout   
                1  2024-03-28T11:40:32.320066992+08:00   stdout   
                
                                                 _pod_name_    _namespace_  \
                0  insights-kafka-consumer-756794579f-5s7c7  arms-apm-demo   
                1  insights-kafka-consumer-756794579f-5s7c7  arms-apm-demo   
                
                                              _pod_uid_ _container_ip_  \
                0  bf30064c-ecc2-414f-8c9c-0718bc36c59e     10.0.0.179   
                1  bf30064c-ecc2-414f-8c9c-0718bc36c59e     10.0.0.179   
                
                                                        _image_name_         _container_name_  \
                0  sy-demo-registry.cn-hangzhou.cr.aliyuncs.com/i...  insights-kafka-consumer   
                1  sy-demo-registry.cn-hangzhou.cr.aliyuncs.com/i...  insights-kafka-consumer   
                
                  __topic__ __source__     __tag__:__pack_id__ __tag__:_node_ip_  \
                0            10.0.0.70  10B250D395188C7E-25C88         10.0.0.70   
                1            10.0.0.70  10B250D395188C7E-25C88         10.0.0.70   
                
                                __tag__:_cluster_id_      __tag__:__hostname__  \
                0  cb5e46cc1148c44c4b33249fb5e6e91d7  cn-zhangjiakou.10.0.0.70   
                1  cb5e46cc1148c44c4b33249fb5e6e91d7  cn-zhangjiakou.10.0.0.70   
                
                        __tag__:_node_name_ __tag__:__receive_time__    __time__  \
                0  cn-zhangjiakou.10.0.0.70               1711597233  1711597233   
                1  cn-zhangjiakou.10.0.0.70               1711597233  1711597233   
                
                                           trace_id           span_id  
                0  f1a04c866767daa41d32054167b13732  9e61f06265a92b7f  
                1  f1a04c866767daa41d32054167b13732  9e61f06265a92b7f  
            '''
            log_df_merge_list.append(pd.read_parquet(log_file_name))

        df_merge = pd.concat(df_merge_list)
        log_df_merge = pd.concat(log_df_merge_list)

        return df_merge, log_df_merge

    def filter_trace_rows(self, df_merge: pd.DataFrame, row: pd.Series) -> pd.DataFrame:
        """
        只保留当前 fault 对应的 trace_id，并裁出后续需要的列。
        """
        trace_id_list = self.parse_list_string(row["trace_id_list"])

        df_merge = df_merge[df_merge["TraceID"].isin(trace_id_list)][
            [
                "Duration", "ResultCode", "Timestamp", "ServiceName", "OperationName",
                "ServiceIp", "TraceID", "SpanId", "ParentSpanId"
            ]
        ].copy()

        df_merge["RpcName"] = df_merge["ServiceName"] + ": " + df_merge["OperationName"]
        return df_merge

    @staticmethod
    def round_to_5_second_bucket(timestamp_ms: int) -> int:
        """
        原代码逻辑：
        1. 毫秒 → 秒
        2. 对齐到 5 秒边界（尾数变成 0 或 5）
        """
        center_timestamp = int(str(timestamp_ms)[:-3])
        center_timestamp -= (
            center_timestamp % 5
            if center_timestamp % 10 < 5
            else -(center_timestamp % 5) + 5
        )
        return center_timestamp

    def add_time_window_columns(self, df_merge: pd.DataFrame) -> pd.DataFrame:
        """
        为每个 span 增加：
        - center_timestamp
        - start_time
        - end_time
        """
        df_merge = df_merge.copy()

        df_merge["center_timestamp"] = (df_merge["Timestamp"] // 10**3).astype(str)
        # 对齐到 5 秒粒度
        df_merge["center_timestamp"] = (
            df_merge["center_timestamp"].str.slice(0, -1)
            + df_merge["center_timestamp"].str[-1].apply(lambda x: "0" if int(x) < 5 else "5")
        )

        df_merge["start_time"] = df_merge["center_timestamp"].astype(int) - (self.step_span * 10)
        df_merge["end_time"] = df_merge["center_timestamp"].astype(int) + (self.step_span * 10)

        return df_merge

    def prefilter_metrics_and_logs(
        self,
        df_merge: pd.DataFrame,
        ip_metrics_df: pd.DataFrame,
        log_df_merge: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        先做一次粗过滤，减少后续逐行 apply 的检索范围。
        """
        filtered_ip_metrics_df = ip_metrics_df[
            ip_metrics_df["IpAddress"].isin(df_merge["ServiceIp"])
            & ip_metrics_df["timestamp"].between(
                df_merge["start_time"].min(),
                df_merge["end_time"].max()
            )
        ]

        filtered_log_df = log_df_merge[
            log_df_merge["trace_id"].isin(df_merge["TraceID"])
            & log_df_merge["span_id"].isin(df_merge["SpanId"])
        ]

        return filtered_ip_metrics_df, filtered_log_df

    def process_single_span_row(
        self,
        row: pd.Series,
        ip_metrics_df: pd.DataFrame,
        log_df_merge: pd.DataFrame,
    ) -> pd.Series:
        """
        对单个 span 进行多模态对齐：
        - 根据 ServiceIp + 时间窗口取 metrics
        - 根据 TraceID + SpanId 取 logs
        """
        # 一个span构建一个时间窗口
        center_timestamp = self.round_to_5_second_bucket(row["Timestamp"])
        start_time = center_timestamp - (self.step_span * 10)
        end_time = center_timestamp + (self.step_span * 10)

        # 1. 对齐 metrics 
        # 筛选ip相同，时间在span窗口内的metric数据
        cond_ip = (
            (ip_metrics_df["IpAddress"] == row["ServiceIp"])
            & (ip_metrics_df["timestamp"] >= start_time)
            & (ip_metrics_df["timestamp"] <= end_time)
        )
        ip_filter = ip_metrics_df[cond_ip]

        for col_name in ip_metrics_df.columns:
            if col_name not in self.outer_name:
                row[col_name] = ip_filter[col_name].to_list()

        # 2. 对齐 logs
        # 筛选同一个 trace + 同一个 span的日志数据
        cond_log = (
            (log_df_merge["trace_id"] == row["TraceID"])
            & (log_df_merge["span_id"] == row["SpanId"])
        )
        log_filter = log_df_merge[cond_log]
        row["logs"] = log_filter["content"].to_list()
        return row

    def align_one_fault_record(self, idx: int, row: pd.Series) -> str:
        """
        对单条 label/fault 记录执行完整对齐。
        如果中间 parquet 已存在，则直接复用。
        fault            start_time    end_time     file_name                                  
        CPU overload     (s)             (s)      "['./trace/file1.parquet', './trace/file2.parquet']"
        
        fault_id    trace_id_list                       count            span_len
        0           "['traceA', 'traceB', 'traceC']"    trace_id 的数量   每个 trace 的长度（span数量）
        """
        output_path = os.path.join(self.data_dir, f"data_alignment_idx_{idx}.parquet")

        if os.path.exists(output_path):
            return output_path

        if self.ip_metrics_df is None:
            raise ValueError("Please run load_inputs() first.")

        # 读取 trace/log 数据
        df_merge, log_df_merge = self.build_trace_and_log_tables(row)

        # 通过trace_id只保留当前 fault 的 trace
        '''
             Duration ResultCode      Timestamp   ServiceName  \
             832          1  1711597003087  mall-gateway   
             830          1  1711597003087  mall-gateway   
            
                                    OperationName   ServiceIp  \
             /components/api/v1/mall/user_info  10.0.0.176   
             MallController.userInfo  10.0.0.176   
            
                   TraceID            SpanId      ParentSpanId  \
             1135397cbd306f46bb3f1b88909e4068  cc805cbfdfc20c2d                 0   
             1135397cbd306f46bb3f1b88909e4068  d065bdae4642e030  cc805cbfdfc20c2d   
            
                                                     RpcName  
             mall-gateway: /components/api/v1/mall/user_info  
             mall-gateway: MallController.userInfo  
        '''
        df_merge = self.filter_trace_rows(df_merge, row)

        # 为每个span增加时间窗口列，以timestamp对齐5秒格作为中心点，前后各扩展10s作为窗口启末
        df_merge = self.add_time_window_columns(df_merge)

        # 先缩小 metrics/log 的候选范围
        # 根据相关 IP + 时间范围过滤metric
        # 根据相关 trace_id + span_id过滤日志
        filtered_ip_metrics_df, filtered_log_df = self.prefilter_metrics_and_logs(
            df_merge, self.ip_metrics_df, log_df_merge
        )

        # 利用每一条span，去筛选trace和span一致的log数据，以及ip一致在时间窗内的metric数据
        df_merge = df_merge.apply(
            lambda span_row: self.process_single_span_row(
                span_row,
                filtered_ip_metrics_df,
                filtered_log_df,
            ),
            axis=1,
        )

        # 得到一个宽表，每一行是一个span对应的数据
        # | TraceID | SpanId | ServiceIp | Timestamp | RpcName | host_cpu_xxx | pod_memory_xxx | ... | logs |

        df_merge.to_parquet(output_path, index=False)
        return output_path

    def padding_result(self, result_url_list: List[str]) -> pd.DataFrame:
        """
        把所有 data_alignment_idx_*.parquet 拼起来，
        对空日志进行占位填充。
        """
        # 将不同注入故障的多模态span数据拼接成一个大表
        result_list = [pd.read_parquet(result_url) for result_url in result_url_list]
        df_result = pd.concat(result_list)

        # 找出没有日志的行用"No related logs"占位
        mask = (df_result["logs"].str.len() == 0) | (df_result["logs"].isnull())
        df_result.loc[mask, "logs"] = pd.Series([self.logs_padding] * mask.sum())

        output_path = os.path.join(self.data_dir, "data_alignment_1.parquet")
        df_result.to_parquet(output_path, index=False)
        print(f"save aligned result to: {output_path}")

        return df_result

    def run(self) -> pd.DataFrame:
        if self.label_df is None or self.ip_metrics_df is None:
            self.load_inputs()

        result_url_list = []
        '''
        label_df:
            fault            start_time    end_time     file_name \                                  
            CPU overload     (s)             (s)      "['./trace/file1.parquet', './trace/file2.parquet']"
            Service error    (s)             (s)

            fault_id    trace_id_list                       count            span_len
            0           "['traceA', 'traceB', 'traceC']"    trace_id 的数量   每个 trace 的长度（span数量）
        '''
        for idx, row in tqdm(self.label_df.iterrows(), total=len(self.label_df), postfix="data alignment"):
            output_path = self.align_one_fault_record(idx, row)
            result_url_list.append(output_path)

        print("concat")
        return self.padding_result(result_url_list)

In [15]:
dataset = "train-ticket"   #  "mall"或 "train-ticket"
config = get_config(dataset)

aligner = DataAligner(config)
result_df = aligner.run()

print("\naligned_df shape:", result_df.shape)
print(result_df.head())

100%|██████████| 31/31 [05:41<00:00, 11.00s/it, data alignment]


concat
save aligned result to: ../train-ticket/data_alignment_1.parquet

aligned_df shape: (38240, 51)
   Duration ResultCode      Timestamp              ServiceName  \
0       895         -1  1719302384259       ts-gateway-service   
1       894         -1  1719302384259       ts-gateway-service   
2       894         -1  1719302384259       ts-gateway-service   
3       893         -1  1719302384261  ts-admin-travel-service   
4       892         -1  1719302384261  ts-admin-travel-service   

                                 OperationName   ServiceIp  \
0       /api/v1/admintravelservice/admintravel  10.0.0.102   
1                   FilteringWebHandler.handle  10.0.0.102   
2  POST /api/v1/admintravelservice/admintravel  10.0.0.102   
3       /api/v1/admintravelservice/admintravel   10.2.0.15   
4              AdminTravelController.addTravel   10.2.0.15   

                            TraceID            SpanId      ParentSpanId  \
0  4e4388bfbcff56a9c9600b53384fe56b  56ee2486ad02cc7